# Tag-keyword diagnostic
**ResolveX - M3: Additional analysis**

Compare the saved classifier with one literal tag-keyword rule on the original test set. Then examine performance with and without words from each question's own weak label.

This is a diagnostic after final evaluation. Keep the classifier, taxonomy, review threshold and split unchanged; use no results here for tuning.

## 1. Load and verify the saved experiment

Check the split fingerprint against the saved classifier. Join by question ID and require exactly one match for every row on both sides. Keep only the 24,853 test questions.

In [1]:
from pathlib import Path
import hashlib
import re
import sys
import time

import joblib
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import accuracy_score, f1_score

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "src/models/classification.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook within ResolveX")
sys.path.insert(0, str(ROOT))

from src.data.taxonomy import TAXONOMY
from src.models.classification import load_dataset, predict_ticket

processed = ROOT / "data/processed"
artifact = joblib.load(processed / "classifier.joblib")
split_path = processed / "classification_splits.csv"
fingerprint = hashlib.sha256(split_path.read_bytes()).hexdigest()
if fingerprint != artifact["metadata"]["split_fingerprint"]:
    raise ValueError("Split fingerprint differs from the saved classifier. Stop the diagnostic.")

frame = load_dataset(processed / "ml_dataset.csv")
assignments = pd.read_csv(split_path, dtype=str, keep_default_na=False)
if assignments["Id"].eq("").any() or assignments["Id"].duplicated().any():
    raise ValueError("Split assignments contain empty or duplicate IDs")
if frame["Id"].eq("").any():
    raise ValueError("Dataset contains empty IDs")
joined = frame.merge(assignments, on="Id", how="outer",
                     validate="one_to_one", indicator=True)
if not joined["_merge"].eq("both").all():
    raise ValueError("Dataset and split IDs do not match on both sides")
test = joined.loc[joined["Split"].eq("test")].drop(columns="_merge").reset_index(drop=True)
assert len(test) == 24853, "Unexpected test-set size"
classes = list(artifact["pipeline"].classes_)
assert len(classes) == 6 and set(classes) == set(TAXONOMY)
assert set(test["Label"]) == set(classes)
print("Split fingerprint: matched")
print("One-to-one ID matches:", len(joined))
print("Test questions:", len(test))
print("Saved temperature:", artifact["temperature"])
print("Saved review threshold:", artifact["review_policy"]["threshold"])

Split fingerprint: matched
One-to-one ID matches: 177194
Test questions: 24853
Saved temperature: 0.8000662155523415
Saved review threshold: 0.7492341336823973


## 2. Reuse the classifier

Call `predict_ticket` for each test title and body. Reuse its calibrated confidence and review flag without fitting or changing anything.

In [2]:
started = time.perf_counter()
predictions = []
for position, row in enumerate(test.itertuples(index=False), 1):
    predictions.append(predict_ticket(artifact, row.Title, row.BodyText))
    if position % 5000 == 0:
        print(f"Predicted {position:,} / {len(test):,}", flush=True)

predictions = pd.DataFrame(predictions)
test["ClassifierPrediction"] = predictions["category"]
test["Confidence"] = predictions["confidence"]
test["HumanReview"] = predictions["human_review"]
assert test["ClassifierPrediction"].isin(classes).all()
assert test["Confidence"].between(0, 1).all()
print(f"Predicted all {len(test):,} questions in {time.perf_counter() - started:.1f} seconds.")

Predicted 5,000 / 24,853


Predicted 10,000 / 24,853


Predicted 15,000 / 24,853


Predicted 20,000 / 24,853


Predicted all 24,853 questions in 17.7 seconds.


## 3. Apply the keyword rule

Count distinct matching taxonomy tags for each category in Title + BodyText. Matching ignores case and uses whole words or phrases. Hyphenated tags accept a hyphen or spaces between their parts. Other spellings are not normalized: `Wi-Fi` does not match `wifi`.

Choose the category with the highest count. Return `none` for a tie or no matches. Repeated occurrences of one tag count only once.

In [3]:
def compile_tag(tag):
    phrase = r"(?:-| +)".join(re.escape(part) for part in tag.split("-"))
    return re.compile(r"(?<!\w)" + phrase + r"(?!\w)", flags=re.IGNORECASE)

patterns = {
    category: [compile_tag(tag) for tag in sorted(TAXONOMY[category])]
    for category in classes
}
assert compile_tag("dual-boot").search("DUAL BOOT")
assert compile_tag("dual-boot").search("dual-boot")
assert not compile_tag("wifi").search("Wi-Fi")
assert not compile_tag("root").search("rooted")

keyword_counts = pd.DataFrame({
    category: [sum(pattern.search(text) is not None for pattern in category_patterns)
               for text in test["Text"]]
    for category, category_patterns in patterns.items()
})
highest_count = keyword_counts.max(axis=1)
winner_count = keyword_counts.eq(highest_count, axis=0).sum(axis=1)
unique_winner = highest_count.gt(0) & winner_count.eq(1)
test["RulePrediction"] = keyword_counts.idxmax(axis=1).where(unique_winner, "none")
test["OwnLabelKeyword"] = [
    keyword_counts.at[index, label] > 0
    for index, label in enumerate(test["Label"])
]
test["KeywordGroup"] = test["OwnLabelKeyword"].map({
    True: "Own-label tag present", False: "No own-label tag"
})
print("Rule predictions of none:", int(test["RulePrediction"].eq("none").sum()))
print("No matches:", int(highest_count.eq(0).sum()))
print("Positive-count ties:", int((highest_count.gt(0) & winner_count.gt(1)).sum()))

Rule predictions of none: 5799
No matches: 2079
Positive-count ties: 3720


## 4. Compare results and group composition

Accuracy here means agreement with the existing tag-derived labels. Every `none` prediction counts as incorrect. Macro F1 always averages the same six categories, with `zero_division=0`; `none` is not a seventh category.

The groups use each question's existing weak label. A question with no own-label tag may still contain tags from other categories or clear wording related to its issue. Grouping is for analysis only.

In [4]:
groups = {
    "All test": test,
    "Own-label tag present": test.loc[test["OwnLabelKeyword"]],
    "No own-label tag": test.loc[~test["OwnLabelKeyword"]],
}
assert sum(len(groups[name]) for name in ("Own-label tag present", "No own-label tag")) == len(test)
assert all(not part.empty for part in groups.values()), "A diagnostic group is empty"

def summarize(part):
    return {
        "Questions": len(part),
        "Test share (%)": 100 * len(part) / len(test),
        "Classifier accuracy (%)": 100 * accuracy_score(part["Label"], part["ClassifierPrediction"]),
        "Classifier Macro F1": f1_score(part["Label"], part["ClassifierPrediction"],
                                        labels=classes, average="macro", zero_division=0),
        "Rule accuracy (%)": 100 * accuracy_score(part["Label"], part["RulePrediction"]),
        "Rule Macro F1": f1_score(part["Label"], part["RulePrediction"],
                                  labels=classes, average="macro", zero_division=0),
        "Classifier review (%)": 100 * part["HumanReview"].mean(),
        "Rule none": int(part["RulePrediction"].eq("none").sum()),
    }

summary = pd.DataFrame.from_dict({name: summarize(part) for name, part in groups.items()}, orient="index")
summary.index.name = "Group"
assert summary.loc["No own-label tag", "Rule accuracy (%)"] == 0
with pd.option_context("display.max_columns", None, "display.width", 220):
    display(summary.round({
        "Test share (%)": 2, "Classifier accuracy (%)": 2,
        "Classifier Macro F1": 4, "Rule accuracy (%)": 2,
        "Rule Macro F1": 4, "Classifier review (%)": 2,
    }))

class_counts = pd.crosstab(test["Label"], test["KeywordGroup"]).reindex(
    index=classes, columns=["Own-label tag present", "No own-label tag"], fill_value=0
)
class_shares = class_counts.div(class_counts.sum(axis=0), axis=1).mul(100)
class_mix = pd.concat({"Questions": class_counts, "Within-group share (%)": class_shares.round(2)}, axis=1)
display(class_mix)

,Questions,Test share (%),Classifier accuracy (%),Classifier Macro F1,Rule accuracy (%),Rule Macro F1,Classifier review (%),Rule none
Group,,,,,,,,
All test,24853,100.0,89.68,0.8856,66.13,0.7325,16.41,5799
Own-label tag present,21423,86.2,93.05,0.9225,76.72,0.8231,11.77,3484
No own-label tag,3430,13.8,68.63,0.6463,0.00,0.0000,45.42,2315


Questions                   \
KeywordGroup           Own-label tag present No own-label tag   
Label                                                           
Access & Permissions                    1819              305   
Boot & Startup                          3275              385   
Hardware & Drivers                      5854              585   
Network & Connectivity                  3956              520   
Software & Packages                     4476             1170   
Storage & Filesystems                   2043              465   

                       Within-group share (%)                   
KeywordGroup            Own-label tag present No own-label tag  
Label                                                           
Access & Permissions                     8.49             8.89  
Boot & Startup                          15.29            11.22  
Hardware & Drivers                      27.33            17.06  
Network & Connectivity                  18.47            15.16  
Software & Packages                     20.89            34.11  
Storage & Filesystems                    9.54            13.56

## 5. Inspect ten examples without an own-label tag

Draw ten questions with seed 42 and show the saved classifier's predictions. Keep their labels and all model settings unchanged.

In [5]:
without_keyword = groups["No own-label tag"]
assert len(without_keyword) >= 10, "Fewer than ten questions in this group"
examples = without_keyword.sample(n=10, random_state=42)[[
    "Id", "Title", "Label", "ClassifierPrediction", "Confidence", "HumanReview"
]].rename(columns={"Label": "Weak label"})
with pd.option_context("display.max_colwidth", 100, "display.max_columns", None, "display.width", 220):
    display(examples.round({"Confidence": 4}))

,Id,Title,Weak label,ClassifierPrediction,Confidence,HumanReview
5139,1262874,System running very slow,Boot & Startup,Software & Packages,0.3988,True
14857,43764,How do I get turbo boost working?,Hardware & Drivers,Hardware & Drivers,0.7451,True
7326,1368879,"How to disable windows key, to not pop out the Advanced MATE Menu?",Hardware & Drivers,Hardware & Drivers,0.9247,False
20103,74488,I can't seem to get any application indicators to show up - why not?,Software & Packages,Software & Packages,0.9336,False
15214,458155,How can I install ubuntu one after upgrading to ubuntu 14.04?,Software & Packages,Software & Packages,0.9967,False
9345,1463684,"Was formatting my flash drive then I closed the process, now my flash drive isn’t showing up, wh...",Storage & Filesystems,Storage & Filesystems,0.7617,False
8809,1439965,WSL2 export is huge - 118 GB!,Storage & Filesystems,Storage & Filesystems,0.4365,True
16440,530265,My blutooth device not work in KUBUNTU,Hardware & Drivers,Hardware & Drivers,0.7289,True
16015,505973,Two Ubuntu Machines cannot access each other's shares anymore,Network & Connectivity,Network & Connectivity,0.9730,False
6778,1343645,How do I hide singularity squashfs partitions in Nautilus,Storage & Filesystems,Storage & Filesystems,0.9364,False


### Notes on the ten examples

The full titles and bodies of these same ten questions were read in an AI-assisted qualitative review. Three weak labels need caution:

| ID | Weak label | Observation |
| --- | --- | --- |
| 1262874 | Boot & Startup | The text describes general slowness and a slow installation, without a boot failure. The Boot label is weakly supported; the model's Software prediction is not automatically verified either. |
| 1368879 | Hardware & Drivers | The request concerns a MATE menu shortcut triggered by the Windows key. The keyboard tag fits the broad mapping, but desktop configuration is not clearly a hardware or driver fault. |
| 530265 | Hardware & Drivers | Bluetooth discovery fails. Hardware/driver and connectivity categories overlap; the text does not establish a driver fault. |

The other seven labels are broadly plausible for the reported requests. Several examples show the limits of literal matching: `install`/`upgraded` do not match `installation`/`upgrade`, `partitions` does not match `partitioning`, and `flash drive` does not match `usb-drive`.

Having no own-label tag match does not mean the question lacks clear topic words. This small review does not estimate label-noise rates. Keep all labels, predictions and model settings unchanged.

## 6. Findings

On the original test set, the classifier agrees with the **tag-derived labels on 89.68%** of questions, compared with **66.13%** for the literal keyword rule. Among the **3,430 questions with no literal match to a tag from their own weak-label category**, classifier agreement is **68.63%**, and **45.42%** receive an additional human-review flag. The flag marks a prediction for extra attention; it does not itself perform a handoff to a person.

The differences below are classifier minus keyword rule. Agreement differences are percentage points; Macro F1 differences use the 0-1 scale.

- **All test:** agreement 89.68% versus 66.13% (**+23.55 percentage points**); Macro F1 0.8856 versus 0.7325 (**+0.1531**).
- **Own-label tag present:** agreement 93.05% versus 76.72% (**+16.34 percentage points**); Macro F1 0.9225 versus 0.8231 (**+0.0993**).
- **No own-label tag:** agreement 68.63% versus 0.00% (**+68.63 percentage points**); Macro F1 0.6463 versus 0.0000 (**+0.6463**).

The rule has zero accuracy in the no-own-label-tag group by construction: it can only choose another category or return none.

The groups may differ in class mix, difficulty and label noise, so performance differences do not establish whether the classifier understands the text.

This literal rule represents one keyword baseline; the results should not be generalized to all keyword-based approaches.